# 🏆 Multi-Model Comparison for Crypto News Classification

Bandingkan 4 model untuk klasifikasi berita crypto:
- **BERT** (bert-base-uncased) - Baseline
- **DistilBERT** (distilbert-base-uncased) - Cepat & ringan
- **RoBERTa** (roberta-base) - Akurasi tertinggi
- **ALBERT** (albert-base-v2) - Sangat ringan

---

## 📋 Daftar Isi

1. [Setup & Install](#1-setup--install)
2. [Upload Data](#2-upload-data)
3. [Data Preparation](#3-data-preparation)
4. [Helper Functions](#4-helper-functions)
5. [Model 1: BERT](#5-model-1-bert)
6. [Model 2: DistilBERT](#6-model-2-distilbert)
7. [Model 3: RoBERTa](#7-model-3-roberta)
8. [Model 4: ALBERT](#8-model-4-albert)
9. [Comparison Results](#9-comparison-results)
10. [Download Best Model](#10-download-best-model)

---

<a id='1-setup--install'></a>
## 1. Setup & Install

In [ ]:
#@title 1.1 Install Packages { display-mode: "form" }
!pip install -q torch transformers scikit-learn accelerate

In [ ]:
#@title 1.2 Check GPU { display-mode: "form" }
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("⚠️ GPU tidak tersedia! Runtime → Change runtime type → T4 GPU")

<a id='2-upload-data'></a>
## 2. Upload Data

In [ ]:
#@title 2.1 Upload labeled_articles.jsonl { display-mode: "form" }
from google.colab import files
import os

print("📤 Upload labeled_articles.jsonl:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename != "labeled_articles.jsonl":
        os.rename(filename, "labeled_articles.jsonl")
    print(f"✅ {filename} uploaded")

!wc -l labeled_articles.jsonl

<a id='3-data-preparation'></a>
## 3. Data Preparation

In [ ]:
#@title 3.1 Load & Explore Data { display-mode: "form" }
import json
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

LABEL_MAP = {"NORMAL": 0, "MODERATE": 1, "CRITICAL": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

articles = []
with open("labeled_articles.jsonl") as f:
    for line in f:
        a = json.loads(line.strip())
        if a.get("label") in LABEL_MAP:
            articles.append(a)

print(f"📊 Total: {len(articles)} articles")
dist = Counter([a["label"] for a in articles])
for label in ["NORMAL", "MODERATE", "CRITICAL"]:
    count = dist[label]
    print(f"  {label}: {count} ({100*count/len(articles):.1f}%)")

fig, ax = plt.subplots(figsize=(6, 3))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
ax.bar(["NORMAL", "MODERATE", "CRITICAL"], [dist["NORMAL"], dist["MODERATE"], dist["CRITICAL"]], color=colors)
ax.set_title("Label Distribution")
plt.tight_layout()
plt.show()

In [ ]:
#@title 3.2 Split Data { display-mode: "form" }
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch

def preprocess(article):
    title = article.get("title", "")
    desc = article.get("description", "")
    return f"TITLE: {title} CONTENT: {desc}"[:512]

texts = [preprocess(a) for a in articles]
labels = [LABEL_MAP[a["label"]] for a in articles]

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

<a id='4-helper-functions'></a>
## 4. Helper Functions

In [ ]:
#@title 4.1 Dataset & Training Functions { display-mode: "form" }
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import time

class CryptoNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

def train_model(model, train_loader, val_loader, device, epochs=3, lr=2e-5, patience=2):
    class_counts = [sum(1 for l in train_labels if l == i) for i in range(3)]
    class_weights = torch.tensor([sum(class_counts)/(3*max(c,1)) for c in class_counts]).to(device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*0.2), total_steps)

    best_f1 = 0
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)
            optimizer.zero_grad()
            outputs = model(ids, attention_mask=mask, labels=lbls)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=-1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)

        model.eval()
        all_preds, all_labels = [], []
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                lbls = batch["labels"].to(device)
                outputs = model(ids, attention_mask=mask, labels=lbls)
                val_loss += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=-1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(lbls.cpu().tolist())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")
        print(f"  Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}, Acc: {correct/total:.4f} | Val Loss: {avg_val_loss:.4f}, F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_model.pt")

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ⏹️ Early stopping at epoch {epoch+1}")
                break

    return best_f1

def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)
            outputs = model(ids, attention_mask=mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(lbls.cpu().tolist())

    f1 = f1_score(all_labels, all_preds, average="macro")
    acc = sum(1 for p, l in zip(all_preds, all_labels) if p == l) / len(all_labels)
    precision = sum(1 for p, l in zip(all_preds, all_labels) if p == l and p == 2) / max(sum(1 for p in all_preds if p == 2), 1)
    recall = sum(1 for p, l in zip(all_preds, all_labels) if p == l and l == 2) / max(sum(1 for l in all_labels if l == 2), 1)
    cm = confusion_matrix(all_labels, all_preds)

    return {"f1_macro": f1, "accuracy": acc, "precision_penting": precision, "recall_penting": recall, "confusion_matrix": cm}

print("✅ Helper functions defined (with early stopping + label smoothing)")

<a id='5-model-1-bert'></a>
## 5. Model 1: BERT (bert-base-uncased)

In [ ]:
#@title 5.1 Load BERT { display-mode: "form" }
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("📥 Loading BERT...")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)

MAX_LEN = 256
BATCH = 8

bert_train = CryptoNewsDataset(train_texts, train_labels, bert_tokenizer, MAX_LEN)
bert_val = CryptoNewsDataset(val_texts, val_labels, bert_tokenizer, MAX_LEN)
bert_test = CryptoNewsDataset(test_texts, test_labels, bert_tokenizer, MAX_LEN)

bert_train_loader = DataLoader(bert_train, batch_size=BATCH, shuffle=True)
bert_val_loader = DataLoader(bert_val, batch_size=BATCH)
bert_test_loader = DataLoader(bert_test, batch_size=BATCH)

print(f"✅ BERT loaded on {device}")

In [ ]:
#@title 5.2 Train BERT { display-mode: "form" }
print("🚀 Training BERT...")
t0 = time.time()
bert_best_f1 = train_model(bert_model, bert_train_loader, bert_val_loader, device, epochs=3, lr=2e-5)
bert_time = time.time() - t0
print(f"\n✅ BERT done in {bert_time:.0f}s, Best Val F1: {bert_best_f1:.4f}")

In [ ]:
#@title 5.3 Evaluate BERT { display-mode: "form" }
bert_model.load_state_dict(torch.load("best_model.pt"))
bert_results = evaluate_model(bert_model, bert_test_loader, device)
print("📊 BERT Results:")
for k, v in bert_results.items():
    print(f"  {k}: {v:.4f}")

<a id='6-model-2-distilbert'></a>
## 6. Model 2: DistilBERT (distilbert-base-uncased)

In [ ]:
#@title 6.1 Load DistilBERT { display-mode: "form" }
print("📥 Loading DistilBERT...")
distil_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
distil_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)
distil_model.to(device)

distil_train = CryptoNewsDataset(train_texts, train_labels, distil_tokenizer, MAX_LEN)
distil_val = CryptoNewsDataset(val_texts, val_labels, distil_tokenizer, MAX_LEN)
distil_test = CryptoNewsDataset(test_texts, test_labels, distil_tokenizer, MAX_LEN)

distil_train_loader = DataLoader(distil_train, batch_size=16, shuffle=True)
distil_val_loader = DataLoader(distil_val, batch_size=16)
distil_test_loader = DataLoader(distil_test, batch_size=16)

print(f"✅ DistilBERT loaded (66M params)")

In [ ]:
#@title 6.2 Train DistilBERT { display-mode: "form" }
print("🚀 Training DistilBERT...")
t0 = time.time()
distil_best_f1 = train_model(distil_model, distil_train_loader, distil_val_loader, device, epochs=3, lr=2e-5)
distil_time = time.time() - t0
print(f"\n✅ DistilBERT done in {distil_time:.0f}s, Best Val F1: {distil_best_f1:.4f}")

In [ ]:
#@title 6.3 Evaluate DistilBERT { display-mode: "form" }
distil_model.load_state_dict(torch.load("best_model.pt"))
distil_results = evaluate_model(distil_model, distil_test_loader, device)
print("📊 DistilBERT Results:")
for k, v in distil_results.items():
    print(f"  {k}: {v:.4f}")

<a id='7-model-3-roberta'></a>
## 7. Model 3: RoBERTa (roberta-base)

In [ ]:
#@title 7.1 Load RoBERTa { display-mode: "form" }
print("📥 Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
roberta_model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=3)
roberta_model.to(device)

roberta_train = CryptoNewsDataset(train_texts, train_labels, roberta_tokenizer, MAX_LEN)
roberta_val = CryptoNewsDataset(val_texts, val_labels, roberta_tokenizer, MAX_LEN)
roberta_test = CryptoNewsDataset(test_texts, test_labels, roberta_tokenizer, MAX_LEN)

roberta_train_loader = DataLoader(roberta_train, batch_size=BATCH, shuffle=True)
roberta_val_loader = DataLoader(roberta_val, batch_size=BATCH)
roberta_test_loader = DataLoader(roberta_test, batch_size=BATCH)

print(f"✅ RoBERTa loaded (125M params)")

In [ ]:
#@title 7.2 Train RoBERTa { display-mode: "form" }
print("🚀 Training RoBERTa...")
t0 = time.time()
roberta_best_f1 = train_model(roberta_model, roberta_train_loader, roberta_val_loader, device, epochs=3, lr=2e-5)
roberta_time = time.time() - t0
print(f"\n✅ RoBERTa done in {roberta_time:.0f}s, Best Val F1: {roberta_best_f1:.4f}")

In [ ]:
#@title 7.3 Evaluate RoBERTa { display-mode: "form" }
roberta_model.load_state_dict(torch.load("best_model.pt"))
roberta_results = evaluate_model(roberta_model, roberta_test_loader, device)
print("📊 RoBERTa Results:")
for k, v in roberta_results.items():
    print(f"  {k}: {v:.4f}")

<a id='8-model-4-albert'></a>
## 8. Model 4: ALBERT (albert-base-v2)

In [ ]:
#@title 8.1 Load ALBERT { display-mode: "form" }
print("📥 Loading ALBERT...")
albert_tokenizer = AutoTokenizer.from_pretrained("albert-base-v2")
albert_model = AutoModelForSequenceClassification.from_pretrained("albert-base-v2", num_labels=3)
albert_model.to(device)

albert_train = CryptoNewsDataset(train_texts, train_labels, albert_tokenizer, MAX_LEN)
albert_val = CryptoNewsDataset(val_texts, val_labels, albert_tokenizer, MAX_LEN)
albert_test = CryptoNewsDataset(test_texts, test_labels, albert_tokenizer, MAX_LEN)

albert_train_loader = DataLoader(albert_train, batch_size=16, shuffle=True)
albert_val_loader = DataLoader(albert_val, batch_size=16)
albert_test_loader = DataLoader(albert_test, batch_size=16)

print(f"✅ ALBERT loaded (12M params)")

In [ ]:
#@title 8.2 Train ALBERT { display-mode: "form" }
print("🚀 Training ALBERT...")
t0 = time.time()
albert_best_f1 = train_model(albert_model, albert_train_loader, albert_val_loader, device, epochs=3, lr=2e-5)
albert_time = time.time() - t0
print(f"\n✅ ALBERT done in {albert_time:.0f}s, Best Val F1: {albert_best_f1:.4f}")

In [ ]:
#@title 8.3 Evaluate ALBERT { display-mode: "form" }
albert_model.load_state_dict(torch.load("best_model.pt"))
albert_results = evaluate_model(albert_model, albert_test_loader, device)
print("📊 ALBERT Results:")
for k, v in albert_results.items():
    print(f"  {k}: {v:.4f}")

<a id='9-comparison-results'></a>
## 9. Comparison Results

In [ ]:
#@title 9.1 Compare All Models { display-mode: "form" }
results = {
    "BERT": {**bert_results, "val_f1": bert_best_f1, "time": bert_time},
    "DistilBERT": {**distil_results, "val_f1": distil_best_f1, "time": distil_time},
    "RoBERTa": {**roberta_results, "val_f1": roberta_best_f1, "time": roberta_time},
    "ALBERT": {**albert_results, "val_f1": albert_best_f1, "time": albert_time},
}

print("=" * 80)
print("🏆 MODEL COMPARISON RESULTS")
print("=" * 80)
print(f"{'Model':<15} {'F1 Macro':<10} {'Accuracy':<10} {'P Prec':<10} {'P Recall':<10} {'Time':<10}")
print("-" * 80)

best_model_name = max(results, key=lambda x: results[x]["f1_macro"])

for name, r in results.items():
    marker = " 🏆" if name == best_model_name else ""
    print(f"{name:<15} {r['f1_macro']:<10.4f} {r['accuracy']:<10.4f} {r['precision_penting']:<10.4f} {r['recall_penting']:<10.4f} {r['time']:<10.0f}s{marker}")

print("=" * 80)
print(f"\n🏆 Best Model: {best_model_name} (F1: {results[best_model_name]['f1_macro']:.4f})")

# Save results
with open("results.json", "w") as f:
    json.dump({k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in results.items()}, f, indent=2)

In [ ]:
#@title 9.2 Plot Comparison Chart { display-mode: "form" }
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_names = list(results.keys())
f1_scores = [results[m]["f1_macro"] for m in model_names]
precisions = [results[m]["precision_penting"] for m in model_names]
recalls = [results[m]["recall_penting"] for m in model_names]

colors = ["#3498db" if m != best_model_name else "#e74c3c" for m in model_names]

axes[0].barh(model_names, f1_scores, color=colors)
axes[0].set_xlabel("F1 Macro")
axes[0].set_title("F1 Macro Score")
axes[0].set_xlim(0, 1)
for i, v in enumerate(f1_scores):
    axes[0].text(v + 0.01, i, f"{v:.3f}", va="center")

axes[1].barh(model_names, precisions, color=colors)
axes[1].set_xlabel("Precision")
axes[1].set_title("CRITICAL Precision")
axes[1].set_xlim(0, 1)
for i, v in enumerate(precisions):
    axes[1].text(v + 0.01, i, f"{v:.3f}", va="center")

axes[2].barh(model_names, recalls, color=colors)
axes[2].set_xlabel("Recall")
axes[2].set_title("CRITICAL Recall")
axes[2].set_xlim(0, 1)
for i, v in enumerate(recalls):
    axes[2].text(v + 0.01, i, f"{v:.3f}", va="center")

plt.tight_layout()
plt.savefig("comparison.png", dpi=150)
plt.show()

<a id='10-download-best-model'></a>
## 10. Download Best Model

In [ ]:
#@title 10.1 Download best_model.pt { display-mode: "form" }
from google.colab import files

print(f"📥 Downloading best model: {best_model_name}")
print(f"\n📋 Copy ke: tradingagents/news_classifier/pretrained/best_model.pt")
files.download("best_model.pt")

In [ ]:
#@title 10.2 Download all models + results (Optional) { display-mode: "form" }
#@markdown Download semua model weights dan results
# import shutil
# shutil.copy("best_model.pt", f"{best_model_name.lower()}_model.pt")
# files.download(f"{best_model_name.lower()}_model.pt")
# files.download("results.json")
# files.download("comparison.png")

---

## 📝 Summary

| Step | Status |
|------|--------|
| 1. Setup & Install | ✅ |
| 2. Upload Data | ✅ |
| 3. Data Preparation | ✅ |
| 4. Helper Functions | ✅ |
| 5. Model 1: BERT | ✅ |
| 6. Model 2: DistilBERT | ✅ |
| 7. Model 3: RoBERTa | ✅ |
| 8. Model 4: ALBERT | ✅ |
| 9. Comparison Results | ✅ |
| 10. Download Best Model | ✅ |

### Next Steps

1. Download `best_model.pt` dari Colab
2. Copy ke `tradingagents/news_classifier/pretrained/best_model.pt`
3. Update config.py dengan model name yang sesuai
4. Test inference: `python -m tradingagents.news_classifier.webhook.server`

---

Built with ❤️ for TradingAgents Stellar Hackathon